# 🤖 Build Your AI Clone

In this workshop you'll analyze your own WhatsApp chat with Python and build an AI that texts like you.

Run the cells in order, from top to bottom, with **Shift + Enter**.

## Setup

This cell prepares the code that reads your WhatsApp chat. Just run it, you don't need to change anything.

In [ ]:
import re
import zipfile
from datetime import datetime

# WhatsApp puts this invisible character before media and system messages
INVISIBLE_MARK = "‎"

# A new message starts with a date and a time, for example:
#   iPhone:  [16/3/25, 18:27:06] Name: text
#   Android: 16/11/2026, 16:51 - Name: text
#   English: 11/16/26, 6:27 PM - Name: text
MESSAGE_START = re.compile(
    r"^\[?(\d{1,4})[/.\-](\d{1,2})[/.\-](\d{1,4}),?\s+"   # date
    r"(\d{1,2}):(\d{2})(?::(\d{2}))?"                     # time
    r"\s*([ap])?\.?\s?(?:m\.?)?"                          # optional AM / PM
    r"\]?\s*(?:-\s*)?(.*)$",                              # rest of the line
    re.IGNORECASE,
)


def load_chat_text(filename, content):
    """Return the text of the chat, from a .txt file or from a .zip export."""
    if filename.lower().endswith(".zip"):
        with zipfile.ZipFile(filename) as archive:
            txt_files = [name for name in archive.namelist() if name.lower().endswith(".txt")]
            content = archive.read(txt_files[0])
    return content.decode("utf-8-sig")


def clean_line(line):
    """Remove invisible characters at the start and turn special spaces into normal ones."""
    line = line.replace(" ", " ").replace(" ", " ")
    return line.lstrip(INVISIBLE_MARK)


def find_day_first(raw_messages):
    """Decide if dates are day/month (Spain) or month/day (USA)."""
    for message in raw_messages:
        first, second = message["date_parts"][0], message["date_parts"][1]
        if first > 31:          # year first, like 2026-11-16
            continue
        if first > 12:
            return True
        if second > 12:
            return False
    return True  # can't tell: use day/month


def make_date(date_parts, time_parts, day_first):
    first, second, third = date_parts
    hour, minute, second_of_time, am_pm = time_parts

    if first > 31:              # year first
        year, month, day = first, second, third
    elif day_first:
        day, month, year = first, second, third
    else:
        month, day, year = first, second, third
    if year < 100:
        year += 2000

    if am_pm == "p" and hour < 12:
        hour += 12
    if am_pm == "a" and hour == 12:
        hour = 0

    return datetime(year, month, day, hour, minute, second_of_time)


def read_chat(chat_text):
    """Turn an exported WhatsApp chat into a list of messages."""
    raw_messages = []

    for line in chat_text.split("\n"):
        line = clean_line(line.rstrip("\r"))
        match = MESSAGE_START.match(line)

        if match:
            numbers = match.groups()
            rest = numbers[7]
            raw_messages.append({
                "date_parts": (int(numbers[0]), int(numbers[1]), int(numbers[2])),
                "time_parts": (int(numbers[3]), int(numbers[4]), int(numbers[5] or 0),
                               (numbers[6] or "").lower()),
                "rest": rest,
            })
        elif raw_messages:
            # A line without a date continues the previous message
            raw_messages[-1]["rest"] += "\n" + line

    day_first = find_day_first(raw_messages)
    messages = []

    for raw in raw_messages:
        # Messages without "Name: " are system messages (Android)
        if ": " not in raw["rest"]:
            continue
        author, text = raw["rest"].split(": ", 1)

        # Media and system messages start with the invisible mark (iPhone)
        if text.startswith(INVISIBLE_MARK):
            continue
        # Media on Android looks like <Media omitted>
        if text.startswith("<") and text.endswith(">"):
            continue

        messages.append({
            "date": make_date(raw["date_parts"], raw["time_parts"], day_first),
            "author": author.strip(),
            "text": text.strip(),
        })

    return messages

In [ ]:
!pip install -q emoji

import urllib.request
from collections import Counter

import emoji
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import MaxNLocator

# Free emoji font from Google, so the charts can show emojis
EMOJI_FONT_URL = "https://github.com/google/fonts/raw/main/ofl/notoemoji/NotoEmoji%5Bwght%5D.ttf"
urllib.request.urlretrieve(EMOJI_FONT_URL, "NotoEmoji.ttf")
EMOJI_FONT = font_manager.FontProperties(fname="NotoEmoji.ttf")

# Skin tones, so 👍🏽 counts as 👍
SKIN_TONES = "\U0001F3FB\U0001F3FC\U0001F3FD\U0001F3FE\U0001F3FF"

# Common words that don't say much about how you text
STOP_WORDS = {
    # Spanish
    "a", "al", "algo", "ante", "así", "como", "con", "cual", "cuando", "de", "del",
    "donde", "e", "el", "ella", "ellas", "ellos", "en", "entre", "era", "es", "esa",
    "ese", "eso", "esta", "está", "este", "esto", "fue", "ha", "hay", "la", "las",
    "le", "les", "lo", "los", "me", "mi", "mis", "muy", "más", "ni", "no", "nos",
    "o", "para", "pero", "por", "porque", "que", "qué", "se", "sea", "si", "sí",
    "sin", "sobre", "son", "su", "sus", "también", "te", "ti", "tu", "tus", "tú",
    "u", "un", "una", "uno", "unos", "y", "ya", "yo", "él",
    # English
    "about", "an", "and", "are", "as", "at", "be", "but", "by", "do", "for", "from",
    "have", "he", "her", "him", "his", "i", "if", "in", "is", "it", "its", "just",
    "me", "my", "of", "on", "or", "our", "she", "so", "that", "the", "their", "them",
    "they", "this", "to", "was", "we", "were", "what", "with", "you", "your",
    # Links
    "http", "https", "www", "com",
}


def bar_chart(items, title, emoji_labels=False):
    """Draw a horizontal bar chart from a list of (label, count) pairs."""
    labels = [label for label, count in items][::-1]
    counts = [count for label, count in items][::-1]

    plt.figure(figsize=(8, 0.5 * len(items) + 1))
    plt.barh(labels, counts, color="#25D366")
    plt.title(title)
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))  # counts are whole numbers
    if emoji_labels:
        plt.yticks(fontproperties=EMOJI_FONT, fontsize=16)
    plt.show()

## Step 1 · Upload your chat

First, export one of your WhatsApp chats **without media**:

- **iPhone:** open the chat → tap the name at the top → **Export Chat** → **Without Media**
- **Android:** open the chat → ⋮ → **More** → **Export chat** → **Without media**

Send the file to your laptop, run the cell below and click **Choose Files** to upload it. Both `.txt` and `.zip` files work.

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
chat_text = load_chat_text(filename, uploaded[filename])

## Step 2 · Read the chat

Now we turn the chat into a list of messages. Each message has a `date`, an `author` and a `text`.

In [ ]:
messages = read_chat(chat_text)

if len(messages) == 0:
    print("⚠️ We couldn't read any messages from this file. Ask an officer for help!")
else:
    print(f"✅ Read {len(messages)} messages!\n")
    for message in messages[:5]:
        print(message["date"], "|", message["author"], "|", message["text"])

## Step 3 · Who are you?

Find yourself in the list and type your number.

In [ ]:
authors = []
for message in messages:
    if message["author"] not in authors:
        authors.append(message["author"])

for number, author in enumerate(authors, start=1):
    print(f"{number}. {author}")

choice = int(input("\nWhich number is you? "))
my_name = authors[choice - 1]

my_messages = [message for message in messages if message["author"] == my_name]
print(f"\nHi {my_name}! You sent {len(my_messages)} messages in this chat.")

## Step 4 · Your WhatsApp Wrapped 🎁

### Who texts the most?

In [ ]:
message_counts = Counter(message["author"] for message in messages)

bar_chart(message_counts.most_common(10), "Who texts the most")

### Your top emojis

In [ ]:
emoji_counts = Counter()
for message in my_messages:
    for found in emoji.emoji_list(message["text"]):
        without_tone = "".join(char for char in found["emoji"] if char not in SKIN_TONES)
        emoji_counts[without_tone] += 1

if emoji_counts:
    bar_chart(emoji_counts.most_common(10), "Your top 10 emojis", emoji_labels=True)
else:
    print("You didn't use any emojis in this chat 🤔")

### Your top words

In [ ]:
word_counts = Counter()
for message in my_messages:
    for word in re.findall(r"[^\W\d_]+", message["text"].lower()):
        if word not in STOP_WORDS:
            word_counts[word] += 1

if word_counts:
    bar_chart(word_counts.most_common(10), "Your top 10 words")
else:
    print("No words found in your messages 🤔")

## Step 5 · Your Style DNA 🧬

Now we turn your stats into a profile of how you text. This is what the AI will use to imitate you.

In [ ]:
# Ways of laughing we look for
LAUGHS = {
    "jaja": r"\b(?:j+a+){2,}\w*",
    "jeje": r"\b(?:j+e+){2,}\w*",
    "jsjs": r"\b(?:j+s+){2,}\w*",
    "haha": r"\b(?:h+a+){2,}\w*",
    "lol": r"\blo+l\b",
    "xd": r"\bxd+\b",
}


def percent(count, total):
    return round(100 * count / total) if total else 0


texts = [message["text"] for message in my_messages]
total = len(texts)

# Message length
average_words = sum(len(text.split()) for text in texts) / total

# Favorite laugh
laugh_counts = Counter()
for text in texts:
    for laugh, pattern in LAUGHS.items():
        laugh_counts[laugh] += len(re.findall(pattern, text.lower()))
favorite_laugh = laugh_counts.most_common(1)[0][0] if sum(laugh_counts.values()) else None

# Capital letters and punctuation
starts_lowercase = sum(1 for text in texts if text[:1].islower())
ends_with_period = sum(1 for text in texts if text.endswith("."))
double_punctuation = sum(1 for text in texts if "??" in text or "!!" in text)

# Messages in a row: how many messages you send each time it's your turn
turns = 0
previous_author = None
for message in messages:
    if message["author"] == my_name and previous_author != my_name:
        turns += 1
    previous_author = message["author"]

style = {
    "average_words": round(average_words, 1),
    "favorite_laugh": favorite_laugh,
    "starts_lowercase": percent(starts_lowercase, total),
    "ends_with_period": percent(ends_with_period, total),
    "double_punctuation": percent(double_punctuation, total),
    "messages_in_a_row": round(total / turns, 1) if turns else 1,
    "top_emojis": [item for item, count in emoji_counts.most_common(5)],
    "top_words": [word for word, count in word_counts.most_common(5)],
}

if style["average_words"] < 5:
    length = "short"
elif style["average_words"] < 12:
    length = "medium"
else:
    length = "long"

print("🧬 Your Style DNA\n")
print(f"📏 Message length: {length} ({style['average_words']} words on average)")
print(f"😂 Favorite laugh: {style['favorite_laugh'] or 'you never laugh in text 😐'}")
print(f"🔡 Starts in lowercase: {style['starts_lowercase']}% of your messages")
print(f"⏺️ Ends with a period: {style['ends_with_period']}% of your messages")
print(f"‼️ Uses ?? or !!: {style['double_punctuation']}% of your messages")
print(f"💬 Messages in a row: {style['messages_in_a_row']} each time you text")
print(f"❤️ Top emojis: {' '.join(style['top_emojis']) or 'none'}")
print(f"🗣️ Top words: {', '.join(style['top_words']) or 'none'}")

## Step 6 · Pick the messages for your clone

Your clone learns from real messages you sent. Choose how many (up to 100): more messages usually make a better clone, but the AI takes longer to reply.

⚠️ **These messages will be sent to Google's AI (Gemini).** On the free plan, Google may keep them and people at Google may read them. To remove messages you don't want to share, type words they contain (names, topics...) and every message with those words is removed.

In [ ]:
import random

# Messages with at least 3 words say more about how you text
candidates = [message["text"] for message in my_messages if len(message["text"].split()) >= 3]
print(f"You have {len(candidates)} messages with 3 or more words.")

amount = int(input("How many do you want your clone to learn from? (up to 100): "))
amount = min(amount, 100, len(candidates))
examples = random.sample(candidates, amount)

# Remove every message that contains a word you don't want to share
answer = input("\nType words to remove (names, topics...), separated by commas, or just press Enter: ")
blocked_words = [word.strip().lower() for word in answer.split(",") if word.strip()]
examples = [text for text in examples if not any(word in text.lower() for word in blocked_words)]

print(f"\n✅ {len(examples)} messages will be sent to the AI.")

if input("Do you want to see them? (y/n): ").strip().lower() == "y":
    for text in examples:
        print("-", text)

## Step 7 · Create your AI clone 🤖

We give the AI your Style DNA and your example messages, and ask it to reply like you.

In [ ]:
from google.colab import ai


def build_prompt(history, new_message):
    """Write the instructions that tell the AI how to text like you."""
    example_lines = "\n".join(f"- {text}" for text in examples)

    conversation = ""
    for turn in history:
        if isinstance(turn, dict):   # newer versions of Gradio
            speaker = "Friend" if turn["role"] == "user" else my_name
            conversation += f"{speaker}: {turn['content']}\n"
        else:                        # older versions: (friend, clone) pairs
            conversation += f"Friend: {turn[0]}\n{my_name}: {turn[1]}\n"

    return f"""You are the AI clone of {my_name}. Reply to the friend exactly like {my_name} texts on WhatsApp.

How {my_name} texts:
- Messages are {length}, about {style['average_words']} words each
- Favorite laugh: {style['favorite_laugh'] or 'none'}
- Starts in lowercase in {style['starts_lowercase']}% of messages
- Ends with a period in {style['ends_with_period']}% of messages
- Uses ?? or !! in {style['double_punctuation']}% of messages
- Sends about {style['messages_in_a_row']} messages in a row (put each one on a new line)
- Favorite emojis: {' '.join(style['top_emojis']) or 'none'}
- Favorite words: {', '.join(style['top_words']) or 'none'}

Real messages from {my_name}:
{example_lines}

Rules:
- Reply in the same language {my_name} uses in the real messages.
- Copy the length, slang, laughs and emojis. Don't sound like an assistant.
- Only write {my_name}'s reply, nothing else.

Conversation:
{conversation}Friend: {new_message}
{my_name}:"""


def clone_reply(new_message, history=[]):
    return ai.generate_text(build_prompt(history, new_message)).strip()


print(clone_reply("hey what are you doing?"))

## Step 8 · Chat with your clone

Run this cell and open the link that ends in `.gradio.live`. You can send it to your friends: it works for 72 hours, while this notebook is open.

In [ ]:
!pip install -q gradio
import gradio as gr


def respond(message, history):
    return clone_reply(message, history)


gr.ChatInterface(respond, title=f"Chat with {my_name}'s AI clone 🤖").launch(share=True)